In [1]:
#ete4 will allow us to gather taxonomy info for the Acidophiles later
!pip install ete4

In [2]:
import pandas as pd
import os
from ete4 import NCBITaxa

#loading in the AciDB tsv
aciDB = pd.read_csv('/content/drive/MyDrive/Capstone/Data/Acidos/filtered_data.tsv', sep='\t')
print(aciDB.columns)
print(aciDB.head())
print("Imported ", len(aciDB), " rows of data")

Index(['name', 'strains', 'domain', 'phylum', 'access_src', 'access_id',
       'isolated', 'gen_completeness', 'ftp_url'],
      dtype='object')
                          name                strains    domain  \
0  Acetobacter nitrogenifigens  LMG 23498 = DSM 23921  Bacteria   
1   Acidibacillus ferrooxidans                  ITV01  Bacteria   
2   Acidibacillus ferrooxidans                 Huett2  Bacteria   
3   Acidibacillus ferrooxidans                  SLC66  Bacteria   
4  Acidibacillus sulfuroxidans                   Y002  Bacteria   

           phylum access_src        access_id isolated  gen_completeness  \
0  Proteobacteria    GenBank  GCA_000429165.1      yes            100.00   
1      Firmicutes    GenBank  GCA_001516615.1      yes             96.98   
2      Firmicutes    GenBank  GCA_002027205.1      yes             96.54   
3      Firmicutes    GenBank  GCA_001642725.1      yes             95.01   
4      Firmicutes    GenBank  GCA_003144315.1      yes             98.0

In [3]:
#Now that the Acidophiles are imported we need to associate the family for each species.
#This cell removes all non-unique species due to additional strains of the same bacteria in the AciDB list
#This cell also fixes the names that aren't going to be read by ete4

aciDBSorted = aciDB.sort_values(by='gen_completeness', ascending=False)
aciDBUnique = aciDBSorted.drop_duplicates(subset='name', keep='first').copy()
aciDBUnique['clean_name'] = aciDBUnique['name'].str.replace(r'\s*\(Candidatus\)', '', regex=True)
aciDBUnique = aciDBUnique.sort_values(by='clean_name')

#verifying which names had to be cleaned
changedNames = aciDBUnique[aciDBUnique['name'] != aciDBUnique['clean_name']]

if len(changedNames) > 0:
    print("Found ", len(changedNames), " names that needed cleaning:")
    # Display both columns side-by-side to visually confirm the change
    print(changedNames[['name', 'clean_name']])
else:
    print("No names were changed")

print("There are ", len(aciDBUnique), " unique bacteria species from AciDB")
print(aciDBUnique.head())


Found  6  names that needed cleaning:
                                          name                    clean_name
166       Gallionella acididurans (Candidatus)       Gallionella acididurans
264           Solibacter usitatus (Candidatus)           Solibacter usitatus
299         Sulfopaludibacter sp. (Candidatus)         Sulfopaludibacter sp.
300  Sulfotelmatobacter kueseliae (Candidatus)  Sulfotelmatobacter kueseliae
301        Sulfotelmatobacter sp. (Candidatus)        Sulfotelmatobacter sp.
302      Sulfotelmatomonas gaucii (Candidatus)      Sulfotelmatomonas gaucii
There are  138  unique bacteria species from AciDB
                          name                strains    domain  \
0  Acetobacter nitrogenifigens  LMG 23498 = DSM 23921  Bacteria   
1   Acidibacillus ferrooxidans                  ITV01  Bacteria   
4  Acidibacillus sulfuroxidans                   Y002  Bacteria   
5     Acidicaldus organivorans                   DX-1  Bacteria   
6         Acidiferrobacter sp.       

In [4]:
#Time to add the family information into the aciDBUnique frame

family = []
ncbi = NCBITaxa()
nameToTaxID = ncbi.get_name_translator(aciDBUnique['clean_name'].tolist())


#this loops through all the species in aciDBUnique to get the family for each
for species in aciDBUnique['clean_name']:

    #Check if the species exists in the dictionary FIRST
    if species in nameToTaxID:
        taxID = nameToTaxID[species][0]

        #getting the data
        lineage = ncbi.get_lineage(taxID)
        ranks = ncbi.get_rank(lineage)
        names = ncbi.get_taxid_translator(lineage)

        #Search for family
        family_name = "Not Found"
        for tid, rank in ranks.items():
            if rank == 'family':
                family_name = names[tid]
                break
        family.append(family_name)

    else:
        #If it wasn't in the dictionary, add a placeholder
        family.append("NCBI_No_Match")

successCount = 0
for item in family:
    if item != "Not Found" and item != "NCBI_No_Match":
        successCount += 1

aciDBUnique['Family'] = family

print("This code iterated over ", len(family), " species.")
print(f"{successCount} species had a family appended")


This code iterated over  138  species.
122 species had a family appended


In [5]:
#keep rows where Family is NOT "NCBI_No_Match" AND NOT "Not Found"
aciDBCleaned = aciDBUnique[
    (aciDBUnique['Family'] != "NCBI_No_Match") &
    (aciDBUnique['Family'] != "Not Found")
].copy()

print(f"Original unique species: {len(aciDBUnique)}")
print(f"Species with family found: {len(aciDBCleaned)}")

Original unique species: 138
Species with family found: 122


In [6]:
#Grouping the bacteria by family and counting species within each family. Print results with largest to smallest families

familyCount = aciDBCleaned.groupby('Family').size().sort_values(ascending=False)
print(familyCount)

Family
Alicyclobacillaceae                          21
Acidobacteriaceae                            16
Acetobacteraceae                             10
Acidocellaceae                                9
Acidimicrobiaceae                             7
Acidithiobacillaceae                          7
Streptomycetaceae                             5
Nitrospiraceae                                5
Clostridiales Family XVII. Incertae Sedis     4
Methylacidiphilaceae                          3
Beijerinckiaceae                              3
Acidiferrobacteraceae                         2
Ferrovaceae                                   2
Thermodesulfobiaceae                          2
Solibacteraceae                               2
Rhodanobacteraceae                            2
Desulfitobacteriaceae                         2
Candidatus Korobacteraceae                    2
Ectothiorhodospiraceae                        2
Comamonadaceae                                1
Chthonomonadaceae                

In [7]:
#this cell looks at how complete the genomes in the selected family are

topFamily = aciDBCleaned[aciDBCleaned['Family'] == 'Alicyclobacillaceae']
print(topFamily[['name', 'gen_completeness']].sort_values(by='gen_completeness', ascending=False))

                                   name  gen_completeness
126     Alicyclobacillus acidocaldarius            100.00
144         Alicyclobacillus vulcanalis             99.61
124        Alicyclobacillus acidiphilus             99.21
137               Alicyclobacillus mali             99.21
139            Alicyclobacillus pomorum             99.20
128    Alicyclobacillus acidoterrestris             98.82
140           Alicyclobacillus sacchari             98.82
130       Alicyclobacillus ferrooxydans             98.80
129        Alicyclobacillus contaminans             98.80
141       Alicyclobacillus shizuokensis             98.56
131          Alicyclobacillus herbarius             98.56
134       Alicyclobacillus kakegawensis             98.56
135  Alicyclobacillus macrosporangiidus             98.53
143     Alicyclobacillus tengchongensis             98.43
132         Alicyclobacillus hesperidum             98.43
4           Acidibacillus sulfuroxidans             98.08
138           

In [8]:
#This cell removes all the species that don't have a completed genome of >95.0%

alicyclobacillusData = aciDBCleaned[aciDBCleaned['Family'] == 'Alicyclobacillaceae']
alicyclobacillusData = alicyclobacillusData[alicyclobacillusData['gen_completeness'] > 95.0]

print(f"{len(alicyclobacillusData)} species were included in the Alicyclobacillaceae list")
print(alicyclobacillusData)

20 species were included in the Alicyclobacillaceae list
                                   name                  strains    domain  \
1            Acidibacillus ferrooxidans                    ITV01  Bacteria   
4           Acidibacillus sulfuroxidans                     Y002  Bacteria   
124        Alicyclobacillus acidiphilus              NBRC 100859  Bacteria   
126     Alicyclobacillus acidocaldarius                   Tc-4-1  Bacteria   
128    Alicyclobacillus acidoterrestris               ATCC 49025  Bacteria   
129        Alicyclobacillus contaminans                DSM 17975  Bacteria   
130       Alicyclobacillus ferrooxydans                    TC-34  Bacteria   
131          Alicyclobacillus herbarius                DSM 13609  Bacteria   
132         Alicyclobacillus hesperidum                DSM 12489  Bacteria   
134       Alicyclobacillus kakegawensis                   5-A83J  Bacteria   
135  Alicyclobacillus macrosporangiidus                DSM 17980  Bacteria   
137    

In [10]:
#save the selected acidophiles in a .csv
alicyclobacillusData.to_csv('/content/drive/MyDrive/Capstone/Data/Acidos/alicyclobacillusAcidos.csv', index=False, header=False, sep=',')

#save the selected acidophiles accession id in a .txt to get genomes from GenBank later
alicyclobacillusData['access_id'].to_csv('/content/drive/MyDrive/Capstone/Data/Acidos/alicyclobacillusAcidoAccession.txt', index=False, header=False, sep='\n')

print("Successfully exported " + str(len(alicyclobacillusData)) + " species to the two files")

Successfully exported 20 species to the two files
